# DentalCare Account Manager - Colab Version

## Optimized for 90% Resource Efficiency

**Settings:**
- Change `MFA_PHONE_URL` below to your phone API URL
- Default: 50 workers (90% efficiency)

## Steps
1. Set Phone API URL
2. Run Cell 1 (Install)
3. Run Cell 2 (Clone)
4. Run Cell 3 (Start)

In [ ]:
# CHANGE THIS TO YOUR PHONE API URL
MFA_PHONE_URL = "https://hot-friends-study.loca.lt/get-number"

# Settings
PARALLEL_COUNT = 50  # workers
BATCH_COUNT = 0  # 0 = unlimited

print("Config:", {
    "Phone API": MFA_PHONE_URL,
    "Workers": PARALLEL_COUNT,
    "RAM": f"~{PARALLEL_COUNT * 180 / 1024:.1f}GB"
})

In [ ]:
# Install dependencies
!pip install playwright requests -q
!playwright install chromium 2>/dev/null || true
print("Dependencies ready")

In [ ]:
# Clone repo
import os
import subprocess

GITHUB_URL = "https://github.com/sulaiman282/Colab_playwrite_test"
APP_DIR = "/content/colab_app"

if os.path.exists(APP_DIR):
    subprocess.run(["rm", "-rf", APP_DIR], check=False)

subprocess.run(["git", "clone", GITHUB_URL, APP_DIR], check=True)
os.chdir(APP_DIR)
print(f"Cloned to {APP_DIR}")

In [ ]:
# Run the app
import sys
import asyncio

sys.path.insert(0, APP_DIR)

from run import ColabRunner

runner = ColabRunner(
    parallel_count=PARALLEL_COUNT,
    batch_count=BATCH_COUNT,
    phone_api_url=MFA_PHONE_URL
)

print("=" * 50)
print("STARTING ACCOUNT MANAGER")
print("=" * 50)

loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
try:
    loop.run_until_complete(runner.run())
finally:
    loop.close()

In [ ]:
# Download results
import glob
from google.colab import files

results_dir = f"{APP_DIR}/data/results"
files_list = glob.glob(f"{results_dir}/accounts_*.json")

if files_list:
    latest = max(files_list, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No results found")